# Week 3 Day 2 — Machine Learning Prediction API

In [2]:
# 1. Check the project files

from pathlib import Path

project_folder = Path.cwd()

required_files = [
    "production_pipeline.py",
    "final_production_candidate.joblib",
    "nordex_mlflow.db",
]

print("Project folder:", project_folder)

for filename in required_files:
    file_path = project_folder / filename
    print(f"{filename}: {'Found' if file_path.is_file() else 'Missing'}")

Project folder: c:\Users\User\AMDARI Project 1
production_pipeline.py: Found
final_production_candidate.joblib: Found
nordex_mlflow.db: Found


In [3]:
import joblib
import pandas as pd

# Makes your Day 1 module available if the saved object uses its class.
import production_pipeline

model_file = project_folder / "final_production_candidate.joblib"
saved_model = joblib.load(model_file)

possible_datasets = [
    project_folder / "final_modeling_dataset.csv",
    project_folder / "final_modeling_dataset(1).csv",
    project_folder / "final_feature_set.csv",
]

dataset_file = next(
    (path for path in possible_datasets if path.is_file()),
    None
)

if dataset_file is None:
    raise FileNotFoundError(
        "Could not find final_modeling_dataset.csv, "
        "final_modeling_dataset(1).csv, or final_feature_set.csv."
    )

manufacturing_data = pd.read_csv(dataset_file)
required_features = list(saved_model.feature_names_in_)

print("Model type:", type(saved_model).__name__)
print("Dataset:", dataset_file.name)
print("Number of model predictors:", len(required_features))
print("Predictors:", required_features)
print("Dataset rows:", len(manufacturing_data))

missing_features = [
    column
    for column in required_features
    if column not in manufacturing_data.columns
]

if missing_features:
    raise ValueError(
        f"The dataset is missing model predictors: {missing_features}"
    )

Model type: Pipeline
Dataset: final_modeling_dataset.csv
Number of model predictors: 20
Predictors: ['production_rate_per_hour', 'good_units', 'units_produced', 'shift_start_hour', 'estimated_oee_pct', 'downtime_minutes', 'runtime_hours', 'defect_rate_pct', 'experience_runtime_interaction', 'temperature_deviation', 'defect_count', 'shift_name', 'experience_level', 'cycle_time_avg', 'machine_prior_avg_downtime', 'machine_prior_avg_defect_rate', 'machine_id', 'machine_prior_avg_units', 'maintenance_downtime', 'humidity']
Dataset rows: 10117


In [4]:
%%writefile week3_day2_api.py
from contextlib import asynccontextmanager
from pathlib import Path
import math

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict, field_validator

# Available for models that reference the Week 3 Day 1 module.
import production_pipeline  # noqa: F401


PROJECT_FOLDER = Path(__file__).resolve().parent
MODEL_FILE = PROJECT_FOLDER / "final_production_candidate.joblib"

NUMERICAL_FEATURES = [
    "production_rate_per_hour",
    "good_units",
    "units_produced",
    "shift_start_hour",
    "estimated_oee_pct",
    "downtime_minutes",
    "runtime_hours",
    "defect_rate_pct",
    "experience_runtime_interaction",
    "temperature_deviation",
    "defect_count",
    "experience_level",
    "cycle_time_avg",
    "machine_prior_avg_downtime",
    "machine_prior_avg_defect_rate",
    "machine_prior_avg_units",
    "maintenance_downtime",
    "humidity",
]

CATEGORICAL_FEATURES = [
    "shift_name",
    "machine_id",
]

ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

model = None
required_features = None


class ManufacturingRecord(BaseModel):
    """One manufacturing record containing all model predictors."""

    model_config = ConfigDict(extra="forbid")

    production_rate_per_hour: float
    good_units: float
    units_produced: float
    shift_start_hour: float
    estimated_oee_pct: float
    downtime_minutes: float
    runtime_hours: float
    defect_rate_pct: float
    experience_runtime_interaction: float
    temperature_deviation: float
    defect_count: float
    shift_name: str
    experience_level: float
    cycle_time_avg: float
    machine_prior_avg_downtime: float
    machine_prior_avg_defect_rate: float
    machine_id: str
    machine_prior_avg_units: float
    maintenance_downtime: float
    humidity: float

    @field_validator(*NUMERICAL_FEATURES, mode="before")
    @classmethod
    def validate_numeric_feature(cls, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError("A numerical predictor must be a number")

        if not math.isfinite(value):
            raise ValueError("A numerical predictor must be finite")

        return value

    @field_validator(*CATEGORICAL_FEATURES)
    @classmethod
    def validate_categorical_feature(cls, value):
        if not value.strip():
            raise ValueError("A categorical predictor cannot be empty")

        return value


class PredictionRequest(BaseModel):
    record: ManufacturingRecord


class PredictionResponse(BaseModel):
    predicted_shift_efficiency_score: float


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Load the saved pipeline once when the API starts."""
    global model, required_features

    if not MODEL_FILE.is_file():
        raise RuntimeError(f"Model file not found: {MODEL_FILE}")

    model = joblib.load(MODEL_FILE)

    if not hasattr(model, "feature_names_in_"):
        raise RuntimeError(
            "The saved model does not expose feature_names_in_."
        )

    required_features = list(model.feature_names_in_)

    if set(required_features) != set(ALL_FEATURES):
        raise RuntimeError(
            "The API schema does not match the saved model. "
            f"Model predictors: {required_features}"
        )

    yield

    model = None
    required_features = None


app = FastAPI(
    title="NorDex Shift Efficiency Prediction API",
    description=(
        "Week 3 Day 2 API for predicting manufacturing "
        "shift efficiency scores."
    ),
    version="1.0.0",
    lifespan=lifespan,
)


@app.get("/health")
def health():
    """Report whether the prediction model is loaded."""
    return {
        "status": "ready" if model is not None else "unavailable",
        "model_loaded": model is not None,
    }


@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    """Validate one record and return its predicted efficiency score."""
    if model is None or required_features is None:
        raise HTTPException(
            status_code=503,
            detail="Prediction model is unavailable",
        )

    record = request.record.model_dump()

    # Match the feature order expected by the saved pipeline.
    input_data = pd.DataFrame(
        [record],
        columns=required_features,
    )

    try:
        prediction = float(model.predict(input_data)[0])
    except (ValueError, TypeError) as exc:
        raise HTTPException(
            status_code=422,
            detail=str(exc),
        ) from exc

    if not math.isfinite(prediction):
        raise HTTPException(
            status_code=500,
            detail="The model returned an invalid prediction",
        )

    return PredictionResponse(
        predicted_shift_efficiency_score=prediction
    )

Overwriting week3_day2_api.py


In [5]:
import json
import numpy as np

valid_rows = manufacturing_data[required_features].dropna()

if valid_rows.empty:
    raise ValueError(
        "No dataset row has values for all 20 predictors."
    )

sample_row = valid_rows.iloc[0]

sample_record = {}

for column in required_features:
    value = sample_row[column]

    if column in ["shift_name", "machine_id"]:
        sample_record[column] = str(value)
    elif isinstance(value, (int, float, np.integer, np.floating)):
        sample_record[column] = float(value)
    else:
        raise TypeError(
            f"Unexpected value for {column}: {value!r}"
        )

valid_request = {"record": sample_record}

print(json.dumps(valid_request, indent=2))

{
  "record": {
    "production_rate_per_hour": 128.31491712707182,
    "good_units": 909.0,
    "units_produced": 929.0,
    "shift_start_hour": 6.0,
    "estimated_oee_pct": 85.71248059068891,
    "downtime_minutes": 15.39,
    "runtime_hours": 7.24,
    "defect_rate_pct": 2.1528525296017222,
    "experience_runtime_interaction": 50.68,
    "temperature_deviation": 2.6000000000000014,
    "defect_count": 20.0,
    "shift_name": "Morning",
    "experience_level": 7.0,
    "cycle_time_avg": 35.65,
    "machine_prior_avg_downtime": 35.420946517412936,
    "machine_prior_avg_defect_rate": 3.4707581118267865,
    "machine_id": "MC_001",
    "machine_prior_avg_units": 661.3048457087752,
    "maintenance_downtime": 0.0,
    "humidity": 49.9
  }
}


In [6]:
sample_dataframe = pd.DataFrame(
    [sample_record],
    columns=required_features,
)

local_prediction = float(
    saved_model.predict(sample_dataframe)[0]
)

print("Local pipeline prediction:", local_prediction)

Local pipeline prediction: 95.74717347801939


In [7]:
from fastapi.testclient import TestClient
from week3_day2_api import app

with TestClient(app) as client:
    health_response = client.get("/health")

print("Status code:", health_response.status_code)
print("Response:", health_response.json())

assert health_response.status_code == 200
assert health_response.json()["model_loaded"] is True

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Status code: 200
Response: {'status': 'ready', 'model_loaded': True}


In [8]:
with TestClient(app) as client:
    valid_response = client.post(
        "/predict",
        json=valid_request,
    )

print("Status code:", valid_response.status_code)
print("Response:", valid_response.json())

assert valid_response.status_code == 200

api_prediction = valid_response.json()[
    "predicted_shift_efficiency_score"
]

assert abs(api_prediction - local_prediction) < 1e-9

print("API matches the saved pipeline:", True)

Status code: 200
Response: {'predicted_shift_efficiency_score': 95.74717347801939}
API matches the saved pipeline: True


In [9]:
missing_column_request = {
    "record": sample_record.copy()
}

del missing_column_request["record"][
    "production_rate_per_hour"
]

with TestClient(app) as client:
    missing_response = client.post(
        "/predict",
        json=missing_column_request,
    )

print("Status code:", missing_response.status_code)
print("Response:", missing_response.json())

assert missing_response.status_code == 422

Status code: 422
Response: {'detail': [{'type': 'missing', 'loc': ['body', 'record', 'production_rate_per_hour'], 'msg': 'Field required', 'input': {'good_units': 909.0, 'units_produced': 929.0, 'shift_start_hour': 6.0, 'estimated_oee_pct': 85.71248059068891, 'downtime_minutes': 15.39, 'runtime_hours': 7.24, 'defect_rate_pct': 2.1528525296017222, 'experience_runtime_interaction': 50.68, 'temperature_deviation': 2.6000000000000014, 'defect_count': 20.0, 'shift_name': 'Morning', 'experience_level': 7.0, 'cycle_time_avg': 35.65, 'machine_prior_avg_downtime': 35.420946517412936, 'machine_prior_avg_defect_rate': 3.4707581118267865, 'machine_id': 'MC_001', 'machine_prior_avg_units': 661.3048457087752, 'maintenance_downtime': 0.0, 'humidity': 49.9}}]}


In [10]:
invalid_number_request = {
    "record": sample_record.copy()
}

invalid_number_request["record"][
    "production_rate_per_hour"
] = "not-a-number"

with TestClient(app) as client:
    invalid_number_response = client.post(
        "/predict",
        json=invalid_number_request,
    )

print("Status code:", invalid_number_response.status_code)
print("Response:", invalid_number_response.json())

assert invalid_number_response.status_code == 422

Status code: 422
Response: {'detail': [{'type': 'value_error', 'loc': ['body', 'record', 'production_rate_per_hour'], 'msg': 'Value error, A numerical predictor must be a number', 'input': 'not-a-number', 'ctx': {'error': {}}}]}


In [11]:
unseen_category_request = {
    "record": sample_record.copy()
}

unseen_category_request["record"][
    "shift_name"
] = "Previously Unseen Shift"

with TestClient(app) as client:
    unseen_response = client.post(
        "/predict",
        json=unseen_category_request,
    )

print("Status code:", unseen_response.status_code)
print("Response:", unseen_response.json())

assert unseen_response.status_code == 200
assert isinstance(
    unseen_response.json()[
        "predicted_shift_efficiency_score"
    ],
    (int, float),
)

Status code: 200
Response: {'predicted_shift_efficiency_score': 94.73005854775823}


In [12]:
extra_column_request = {
    "record": sample_record.copy()
}

extra_column_request["record"][
    "unexpected_column"
] = 123

with TestClient(app) as client:
    extra_response = client.post(
        "/predict",
        json=extra_column_request,
    )

print("Status code:", extra_response.status_code)
print("Response:", extra_response.json())

assert extra_response.status_code == 422

Status code: 422
Response: {'detail': [{'type': 'extra_forbidden', 'loc': ['body', 'record', 'unexpected_column'], 'msg': 'Extra inputs are not permitted', 'input': 123}]}


In [13]:
test_results = pd.DataFrame([
    {
        "Test": "Health endpoint",
        "Expected status": 200,
        "Actual status": health_response.status_code,
        "Result": (
            "Passed"
            if health_response.status_code == 200
            else "Failed"
        ),
    },
    {
        "Test": "Valid prediction",
        "Expected status": 200,
        "Actual status": valid_response.status_code,
        "Result": (
            "Passed"
            if valid_response.status_code == 200
            and abs(api_prediction - local_prediction) < 1e-9
            else "Failed"
        ),
    },
    {
        "Test": "Missing required predictor",
        "Expected status": 422,
        "Actual status": missing_response.status_code,
        "Result": (
            "Passed"
            if missing_response.status_code == 422
            else "Failed"
        ),
    },
    {
        "Test": "Nonnumeric predictor",
        "Expected status": 422,
        "Actual status": invalid_number_response.status_code,
        "Result": (
            "Passed"
            if invalid_number_response.status_code == 422
            else "Failed"
        ),
    },
    {
        "Test": "Unseen shift category",
        "Expected status": 200,
        "Actual status": unseen_response.status_code,
        "Result": (
            "Passed"
            if unseen_response.status_code == 200
            else "Failed"
        ),
    },
    {
        "Test": "Unexpected predictor",
        "Expected status": 422,
        "Actual status": extra_response.status_code,
        "Result": (
            "Passed"
            if extra_response.status_code == 422
            else "Failed"
        ),
    },
])

testing_report_file = (
    project_folder
    / "week3_day2_prediction_endpoint_testing_results.csv"
)

test_results.to_csv(
    testing_report_file,
    index=False,
)

display(test_results)
print("Saved:", testing_report_file.name)

,Test,Expected status,Actual status,Result
0,Health endpoint,200,200,Passed
1,Valid prediction,200,200,Passed
2,Missing required predictor,422,422,Passed
3,Nonnumeric predictor,422,422,Passed
4,Unseen shift category,200,200,Passed
5,Unexpected predictor,422,422,Passed


Saved: week3_day2_prediction_endpoint_testing_results.csv


## API usage

### GET /health

Checks whether the saved production model has loaded.

Successful response: HTTP 200 with `status` and `model_loaded`.

### POST /predict

Accepts a JSON object containing one `record` with the 20 manufacturing predictors.

Successful response: HTTP 200 with `predicted_shift_efficiency_score`.

Invalid request: HTTP 422, including details of missing, invalid, or unexpected predictors.

### Model workflow

The service loads `final_production_candidate.joblib` when it starts.
Incoming records are validated using Pydantic, placed in the saved model's
feature order, and passed to the pipeline's `predict()` method. The pipeline
uses its saved preprocessing before generating a Shift Efficiency Score.

In [14]:
deliverables = [
    project_folder / "week3_day2_prediction_api.ipynb",
    project_folder / "week3_day2_api.py",
    project_folder / (
        "week3_day2_prediction_endpoint_testing_results.csv"
    ),
]

deliverable_check = pd.DataFrame({
    "Deliverable": [path.name for path in deliverables],
    "Exists": [path.is_file() for path in deliverables],
})

display(deliverable_check)

,Deliverable,Exists
0,week3_day2_prediction_api.ipynb,True
1,week3_day2_api.py,True
2,week3_day2_prediction_endpoint_testing_results...,True
